# text classification with knowledge distillation

exploring how knowledge distillation works for sentiment analysis. going to build a big teacher model and see if a tiny student can learn from it.

goals:
- load the imdb movie review dataset
- build teacher (big bidirectional lstm) 
- build student (small simple lstm)
- train both and compare them
- see if distillation actually helps

## a note on the stored outputs

The outputs of this notebook were cleared deliberately.
They were stale, they came from an earlier synthetic-data version of the notebook, and they contradicted both the current source and each other.

What was wrong with them, recorded here so the evidence is not lost along with the outputs:

- Stored output reported `vocabulary size: 62`, while the cell above it reported `vocabulary size: 10000` and the current source loads the 10,000-word IMDB set with `num_words=10000`.
- The two summary cells disagreed about model size. The size comparison cell reported a 2,766,146-parameter teacher and a 675,170-parameter student, the final summary cell reported 1,494,082 and 39,138 for the same two models.
- Teacher, distilled student and control student all reported exactly 100.00% accuracy, which is not a plausible IMDB score for any of them, let alone all three.
- The distillation training loop cell ended in a `KeyboardInterrupt` traceback, so that run never finished.
- Cell execution counts ran out of order (25, then 61 through 68, then 31, 32, 41, 44), so the stored state was never produced by one top-to-bottom pass.

TensorFlow is not installed in the environment this was cleared from and has no wheel for the Python version available there, so the notebook could not simply be re-run.
No source cell was changed. The distillation code is the same maths as the MNIST notebook and is believed sound.

This notebook needs a clean top-to-bottom re-run on a machine with TensorFlow before any number in it is quoted.

## imports and setup

In [ ]:
# using keras instead of pytorch (had issues with torch on this machine)
# uncomment if you need to install
#!pip install tensorflow

import os
# hide tensorflow's c++ start up messages so the saved outputs only show the experiment
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import time
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

# python side tensorflow notices, such as the one about gpus on windows, would otherwise land in the outputs
tf.get_logger().setLevel('ERROR')

print(f"tensorflow version: {tf.__version__}")

In [ ]:
SEED = 0
# fixed seeds alone do not make tensorflow repeatable, deterministic kernels are needed as well
tf.config.experimental.enable_op_determinism()

## loading real imdb movie review dataset

using the famous imdb dataset - 50k movie reviews labeled as positive or negative. way better than fake data!

In [ ]:
# imdb: 25k train and 25k test reviews, binary sentiment, already tokenized to word indices
from tensorflow.keras.datasets import imdb

# keep the 10k most frequent words, rarer ones become the out of vocabulary id 2
max_features = 10000
(x_train_raw, y_train), (x_test_raw, y_test) = imdb.load_data(num_words=max_features)

print(f"training samples: {len(x_train_raw)}")
print(f"test samples: {len(x_test_raw)}")
print(f"classes: {len(set(y_train))} (0=negative, 1=positive)")

In [ ]:
# both splits are balanced, so plain accuracy is a fair metric
train_dist = Counter(y_train)
test_dist = Counter(y_test)
print(f"\ntrain distribution: {dict(train_dist)}")
print(f"test distribution: {dict(test_dist)}")

In [ ]:
# get the word index mapping to convert back to text
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

def decode_review(encoded_review):
    """convert sequence of integers back to text"""
    # load_data shifts word indices by 3: 0 is padding, 1 marks the start, 2 is out of vocabulary
    decoded = ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])
    return decoded

In [ ]:
print("first positive review:")
pos_idx = list(y_train).index(1)
print(f"encoded: {x_train_raw[pos_idx][:20]}...")
print(f"decoded: {decode_review(x_train_raw[pos_idx])[:200]}...")
print(f"label: {y_train[pos_idx]} (positive)")

print("\nfirst negative review:")
neg_idx = list(y_train).index(0)  
print(f"encoded: {x_train_raw[neg_idx][:20]}...")
print(f"decoded: {decode_review(x_train_raw[neg_idx])[:200]}...")
print(f"label: {y_train[neg_idx]} (negative)")

In [ ]:
# pad sequences to same length for batching
from tensorflow.keras.preprocessing.sequence import pad_sequences

train_lengths = [len(seq) for seq in x_train_raw]
test_lengths = [len(seq) for seq in x_test_raw]

print(f"sequence length stats:")
print(f"train - min: {min(train_lengths)}, max: {max(train_lengths)}, avg: {np.mean(train_lengths):.1f}")
print(f"test - min: {min(test_lengths)}, max: {max(test_lengths)}, avg: {np.mean(test_lengths):.1f}")

In [ ]:
# 256 tokens keep the bidirectional teacher trainable on a cpu, reviews longer than that lose their start
max_len = 256
print(f"\nusing max_len: {max_len}")

# zeros go before the review and long reviews keep their last 256 words, so the lstm's final
# state, the only one the classifier sees, comes right after real words instead of after padding
x_train = pad_sequences(x_train_raw, maxlen=max_len, padding='pre', truncating='pre')
x_test = pad_sequences(x_test_raw, maxlen=max_len, padding='pre', truncating='pre')

In [ ]:
print(f"final shapes:")
print(f"x_train: {x_train.shape}")
print(f"x_test: {x_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

# embedding table size: kept word ids run from 0 to max_features - 1
vocab_size = max_features
print(f"vocabulary size: {vocab_size}")

## explore the real dataset

checking out the imdb reviews and their characteristics

In [ ]:
print("positive review examples:")
pos_indices = [i for i, label in enumerate(y_train) if label == 1][:3]
for i, idx in enumerate(pos_indices):
    review = decode_review(x_train_raw[idx])
    print(f"\n{i+1}. {review[:300]}...")

print("\n" + "="*50)
print("negative review examples:")
neg_indices = [i for i, label in enumerate(y_train) if label == 0][:3]
for i, idx in enumerate(neg_indices):
    review = decode_review(x_train_raw[idx])
    print(f"\n{i+1}. {review[:300]}...")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, lengths, name, color in [(axes[0], train_lengths, 'training', 'steelblue'), (axes[1], test_lengths, 'test', 'orange')]:
    ax.hist(lengths, bins=50, edgecolor='black', alpha=0.7, color=color)
    ax.axvline(max_len, color='red', linestyle='--', linewidth=2, label=f'max_len={max_len}')
    ax.set_title(f'{name} sequence lengths', fontweight='bold')
    ax.set_xlabel('sequence length (tokens)')
    ax.set_ylabel('reviews')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# index the counters by label so the bars cannot be swapped by insertion order
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, dist, name in [(axes[0], train_dist, 'training'), (axes[1], test_dist, 'test')]:
    ax.bar(['negative (0)', 'positive (1)'], [dist[0], dist[1]], color=['red', 'green'], alpha=0.7, edgecolor='black')
    ax.set_title(f'{name} class distribution', fontweight='bold')
    ax.set_ylabel('reviews')
    ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
print(f"coverage with max_len={max_len}:")
print(f"train: {sum(1 for l in train_lengths if l <= max_len)/len(train_lengths)*100:.1f}%")
print(f"test: {sum(1 for l in test_lengths if l <= max_len)/len(test_lengths)*100:.1f}%")

In [ ]:
# real token counts over the training reviews, id 1 is the start marker and id 2 the out of vocabulary marker
token_counts = np.bincount(np.concatenate(x_train_raw), minlength=max_features)
top_ids = np.argsort(token_counts[3:])[::-1][:20] + 3
top_words = [reverse_word_index[i - 3] for i in top_ids]

plt.figure(figsize=(12, 6))
plt.barh(range(len(top_words)), token_counts[top_ids], color='steelblue')
plt.yticks(range(len(top_words)), top_words)
plt.xlabel('occurrences in the training reviews')
plt.title('top 20 most frequent words in the imdb training reviews', fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# word ids start at 4, so ids below max_features keep max_features - 4 distinct words
word_tokens = token_counts[3:].sum() + token_counts[2]
print(f"total vocabulary size in imdb: {len(word_index):,} words")
print(f"kept: the {max_features - 4:,} most frequent words, ids 4 to {max_features - 1}")
print(f"training word tokens covered by the kept words: {token_counts[3:].sum() / word_tokens * 100:.1f}%")

## data preprocessing complete

the imdb dataset comes pre-tokenized, so we just need to pad sequences for batching

In [ ]:
# imdb arrives tokenized, so padding was the only preprocessing step
print("data ready for training:")
print(f"x_train shape: {x_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"vocab size: {vocab_size}")
print(f"max sequence length: {max_len}")

example_idx = 0
print(f"\nexample padded sequence (last 20 tokens, the padding sits at the start):")
print(f"original length: {len(x_train_raw[example_idx])}")
print(f"padded sequence: {x_train[example_idx][-20:]}")
print(f"decoded: {decode_review(x_train[example_idx][-20:])}")

hold out 5,000 training reviews as a validation set, everything looked at during training uses it so the test set is only read at the end

In [ ]:
perm = np.random.default_rng(SEED).permutation(len(x_train))
val_idx, train_idx = perm[:5000], perm[5000:]
x_val, y_val = x_train[val_idx], y_train[val_idx]
x_train, y_train = x_train[train_idx], y_train[train_idx]
print(f"train: {len(x_train)}, validation: {len(x_val)}, test: {len(x_test)}")

## build the teacher model

making a big bidirectional lstm with lots of parameters

In [ ]:
def create_teacher_model(vocab_size, max_len):
    model = keras.Sequential([
        keras.layers.Input(shape=(max_len,)),
        keras.layers.Embedding(vocab_size, 128),
        keras.layers.Bidirectional(keras.layers.LSTM(256, return_sequences=True, dropout=0.3)),
        keras.layers.Bidirectional(keras.layers.LSTM(128, dropout=0.3)),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.3),
        # no softmax: the model emits logits so the distillation loss can soften them with softmax(z / t)
        keras.layers.Dense(2)
    ])
    return model

In [ ]:
# teacher: two stacked bidirectional lstms, each review is read in both directions
create_teacher_model(vocab_size, max_len).summary()

## build the student model

way smaller - just one lstm layer and fewer params

In [ ]:
def create_student_model(vocab_size, max_len):
    model = keras.Sequential([
        keras.layers.Input(shape=(max_len,)),
        keras.layers.Embedding(vocab_size, 64),
        keras.layers.LSTM(64, dropout=0.3),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.3),
        # no softmax, as in the teacher
        keras.layers.Dense(2)
    ])
    return model

In [ ]:
# student: one unidirectional lstm and a smaller embedding
create_student_model(vocab_size, max_len).summary()

## compare model sizes

checking how much smaller the student is

In [ ]:
teacher_params = create_teacher_model(vocab_size, max_len).count_params()
student_params = create_student_model(vocab_size, max_len).count_params()

print("model size comparison:")
print(f"teacher: {teacher_params:,} parameters")
print(f"student: {student_params:,} parameters")
print(f"compression ratio: {teacher_params/student_params:.1f}x")
print(f"size reduction: {(1-student_params/teacher_params)*100:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
models = ['teacher\n(bidirectional)', 'student\n(simple)']
params = [teacher_params/1000, student_params/1000]  # in thousands
colors = ['#e74c3c', '#3498db']

bars = ax.bar(models, params, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('parameters (thousands)')
ax.set_title('model size comparison', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, param in zip(bars, params):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{param:.1f}k',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## implement knowledge distillation

the loss is `alpha * soft + (1 - alpha) * hard`: the soft term is `KL(softmax(teacher / T) || softmax(student / T))` times `T^2`, the hard term is the ordinary cross entropy with the labels.
with `alpha = 0` it is plain supervised training, which is how the teacher and the control student are trained.

In [ ]:
def distillation_loss(y_true, student_logits, teacher_logits, temperature=4.0, alpha=0.7):
    """
    hinton style distillation loss: alpha * soft + (1 - alpha) * hard
    alpha is the weight on matching the teacher, 1 - alpha the weight on the true labels
    """
    # hard term at t=1: cross entropy with the true labels
    hard_loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=tf.cast(y_true, tf.int32), logits=student_logits))
    if alpha == 0:
        return hard_loss

    # soft term: kl(teacher_t || student_t) on logits, a temperature above 1 flattens the teacher's
    # distribution so the probability it gives the wrong class carries signal, log space avoids an epsilon
    teacher_log = tf.nn.log_softmax(teacher_logits / temperature)
    student_log = tf.nn.log_softmax(student_logits / temperature)
    soft_loss = tf.reduce_sum(tf.exp(teacher_log) * (teacher_log - student_log), axis=1)
    # times t^2 to undo the 1 / t^2 shrink of the soft target gradients
    soft_loss = tf.reduce_mean(soft_loss) * (temperature ** 2)

    return alpha * soft_loss + (1 - alpha) * hard_loss

one compiled training step, a custom step because `fit()` has no hook for feeding the teacher's outputs into the loss

In [ ]:
def make_train_step(model, optimizer, temperature, alpha):
    @tf.function(reduce_retracing=True)
    def train_step(x, y, teacher_logits):
        with tf.GradientTape() as tape:
            logits = model(x, training=True)
            loss = distillation_loss(y, logits, teacher_logits, temperature, alpha)
        # only this model's variables get gradients, which is what keeps the teacher frozen
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        return loss
    return train_step

In [ ]:
def accuracy(model, x, y):
    # inference mode turns dropout off, argmax of the logits is the same at every temperature
    logits = model.predict(x, batch_size=500, verbose=0)
    return float(np.mean(np.argmax(logits, axis=1) == y))

the same loop trains the teacher, the control student and the distilled student.
for a given seed the two students start from the same weights and see the same shuffled batches, so the loss is the only difference between them

In [ ]:
def train(model, x, y, teacher_logits=None, temperature=1.0, alpha=0.0, epochs=5,
          batch_size=64, seed=0):
    optimizer = keras.optimizers.Adam(learning_rate=0.001)
    train_step = make_train_step(model, optimizer, temperature, alpha)
    # one generator per run, so the batch order depends on the seed and nothing else
    rng = np.random.default_rng(seed)
    history = {'epoch': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        start_time = time.time()
        order = rng.permutation(len(x))
        for start in range(0, len(x), batch_size):
            idx = order[start:start + batch_size]
            batch_teacher = None if teacher_logits is None else teacher_logits[idx]
            train_step(x[idx], y[idx], batch_teacher)
        history['epoch'].append(epoch)
        history['val_acc'].append(accuracy(model, x_val, y_val))
        print(f"  epoch {epoch}/{epochs}: validation accuracy {history['val_acc'][-1]:.4f}, {time.time() - start_time:.0f} s")
    return history

In [ ]:
def new_model(create, seed):
    # the global seed fixes the initial weights and the dropout masks of the model built next
    keras.utils.set_random_seed(seed)
    return create(vocab_size, max_len)

## train the teacher

training the big model first so we can use it to teach the student

In [ ]:
teacher_epochs = 3

In [ ]:
# the teacher is trained first, on hard labels only
t0 = time.time()
teacher = new_model(create_teacher_model, SEED)
teacher_history = train(teacher, x_train, y_train, epochs=teacher_epochs, seed=SEED)
print(f"teacher trained in {(time.time() - t0) / 60:.1f} min")

freeze the teacher and compute its logits for every training review once, with no augmentation these are exactly the logits it would give batch by batch

In [ ]:
teacher.trainable = False
teacher_train_logits = teacher.predict(x_train, batch_size=500, verbose=0)
print(f"teacher logits: {teacher_train_logits.shape}")

## train student with distillation

the distilled student and the control student get the same training set, epochs, batch size and seed

In [ ]:
# t=4 gives moderately soft targets, alpha=0.7 puts most of the weight on the teacher
temperature = 4.0
alpha = 0.7
epochs = 5

In [ ]:
t0 = time.time()
student_distilled = new_model(create_student_model, SEED)
distilled_history = train(student_distilled, x_train, y_train, teacher_logits=teacher_train_logits,
                          temperature=temperature, alpha=alpha, epochs=epochs, seed=SEED)
print(f"distilled student trained in {(time.time() - t0) / 60:.1f} min")

## train student without distillation (control)

the same student, seed and budget, trained on the hard labels alone

In [ ]:
t0 = time.time()
student_baseline = new_model(create_student_model, SEED)
baseline_history = train(student_baseline, x_train, y_train, epochs=epochs, seed=SEED)
print(f"control student trained in {(time.time() - t0) / 60:.1f} min")

## compare all models

validation accuracy after every epoch, then the test set, read once at the end

In [ ]:
RESULTS_DIR = os.path.join('..', 'results', 'text_classification')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(teacher_history['epoch'], np.array(teacher_history['val_acc']) * 100, marker='o', color='#e74c3c', label='teacher')
ax.plot(baseline_history['epoch'], np.array(baseline_history['val_acc']) * 100, marker='^', color='#95a5a6', label='student (control)')
ax.plot(distilled_history['epoch'], np.array(distilled_history['val_acc']) * 100, marker='s', color='#27ae60', label='student (distilled)')
ax.set_title('validation accuracy during training', fontweight='bold')
ax.set_xlabel('epoch')
ax.set_ylabel('accuracy (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'text_distillation_results.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
teacher_acc = accuracy(teacher, x_test, y_test)
baseline_acc = accuracy(student_baseline, x_test, y_test)
distilled_acc = accuracy(student_distilled, x_test, y_test)

In [ ]:
print("test accuracy:")
print(f"teacher: {teacher_acc*100:.2f}%")
print(f"student control: {baseline_acc*100:.2f}%")
print(f"student distilled: {distilled_acc*100:.2f}%")
print(f"\ndistilled minus control: {(distilled_acc - baseline_acc)*100:+.2f} percentage points")

## test on some examples

trying the models on actual text to see how they predict

In [ ]:
# hand written reviews, a quick qualitative check outside the imdb test set
test_examples_raw = [
    "this movie was absolutely fantastic with amazing acting and plot",
    "worst film ever terrible acting boring story waste of time",
    "pretty good movie enjoyed most parts but ending was weak", 
    "not the best but had some decent moments worth watching",
    "horrible cinematography poor dialogue would not recommend",
    "brilliant masterpiece excellent direction superb performances"
]

print("converting test examples to sequences...")

In [ ]:
def text_to_sequence(text, word_index, max_features, index_from=3):
    """map words to the same ids imdb.load_data produces"""
    # load_data starts every review with id 1, shifts word ranks by index_from and
    # replaces ids outside the kept vocabulary with the out of vocabulary id 2
    sequence = [1]
    for word in text.lower().split():
        rank = word_index.get(word)
        if rank is None or rank + index_from >= max_features:
            sequence.append(2)
        else:
            sequence.append(rank + index_from)
    return sequence

In [ ]:
test_sequences = [text_to_sequence(text, word_index, max_features) for text in test_examples_raw]
test_padded = pad_sequences(test_sequences, maxlen=max_len, padding='pre')

In [ ]:
print("test sequences created:")
for i, (text, seq) in enumerate(zip(test_examples_raw, test_sequences)):
    print(f"{i+1}. '{text}' -> {len(seq) - 1} words, {seq.count(2)} out of vocabulary")

In [ ]:
teacher_preds = tf.nn.softmax(teacher.predict(test_padded, verbose=0)).numpy()
baseline_preds = tf.nn.softmax(student_baseline.predict(test_padded, verbose=0)).numpy()
distilled_preds = tf.nn.softmax(student_distilled.predict(test_padded, verbose=0)).numpy()

In [ ]:
print("\npredictions on movie reviews:")
print("="*80)
for i, text in enumerate(test_examples_raw):
    print(f"\nreview: '{text}'")
    print(f"  teacher: {'positive' if np.argmax(teacher_preds[i]) == 1 else 'negative'} (conf: {np.max(teacher_preds[i]):.3f})")
    print(f"  baseline: {'positive' if np.argmax(baseline_preds[i]) == 1 else 'negative'} (conf: {np.max(baseline_preds[i]):.3f})")
    print(f"  distilled: {'positive' if np.argmax(distilled_preds[i]) == 1 else 'negative'} (conf: {np.max(distilled_preds[i]):.3f})")

## analyze prediction confidence

checking how confident each model is in its predictions

In [ ]:
# confidence is the probability of the predicted class
teacher_test_preds = tf.nn.softmax(teacher.predict(x_test, verbose=0)).numpy()
baseline_test_preds = tf.nn.softmax(student_baseline.predict(x_test, verbose=0)).numpy()
distilled_test_preds = tf.nn.softmax(student_distilled.predict(x_test, verbose=0)).numpy()

teacher_conf = np.max(teacher_test_preds, axis=1)
baseline_conf = np.max(baseline_test_preds, axis=1)
distilled_conf = np.max(distilled_test_preds, axis=1)

In [ ]:


fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(teacher_conf, bins=20, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[0].set_title('teacher confidence', fontweight='bold')
axes[0].set_xlabel('confidence')
axes[0].set_ylabel('frequency')
axes[0].axvline(np.mean(teacher_conf), color='black', linestyle='--', linewidth=2, label=f'mean: {np.mean(teacher_conf):.3f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(baseline_conf, bins=20, color='#95a5a6', alpha=0.7, edgecolor='black')
axes[1].set_title('student baseline confidence', fontweight='bold')
axes[1].set_xlabel('confidence')
axes[1].set_ylabel('frequency')
axes[1].axvline(np.mean(baseline_conf), color='black', linestyle='--', linewidth=2, label=f'mean: {np.mean(baseline_conf):.3f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].hist(distilled_conf, bins=20, color='#27ae60', alpha=0.7, edgecolor='black')
axes[2].set_title('student distilled confidence', fontweight='bold')
axes[2].set_xlabel('confidence')
axes[2].set_ylabel('frequency')
axes[2].axvline(np.mean(distilled_conf), color='black', linestyle='--', linewidth=2, label=f'mean: {np.mean(distilled_conf):.3f}')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("average confidence scores:")
print(f"teacher: {np.mean(teacher_conf):.3f}")
print(f"baseline: {np.mean(baseline_conf):.3f}")
print(f"distilled: {np.mean(distilled_conf):.3f}")

## final results

In [ ]:
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"\nmodel sizes:")
print(f"  teacher: {teacher_params:,} params")
print(f"  student: {student_params:,} params")
print(f"  compression: {teacher_params/student_params:.1f}x smaller")
print(f"\ntest accuracy:")
print(f"  teacher: {teacher_acc*100:.2f}%")
print(f"  student control: {baseline_acc*100:.2f}%")
print(f"  student distilled (t={temperature}, alpha={alpha}): {distilled_acc*100:.2f}%")
print(f"  distilled minus control: {(distilled_acc - baseline_acc)*100:+.2f} percentage points")